# Rebuild `psi_ihn_subtypes.csv`

The SPINE feather files are named by the **seed coordinate** of the neuron they describe
(`cell_type_predictions_<x>_<y>_<z>.feather`), not by root ID, so the figure notebooks need
a lookup from coordinate to PSI-IhN population.

This notebook builds it:

1. parse `x_y_z` out of each feather filename;
2. read the segmentation at that voxel to get the root ID (CloudVolume, mip 32×32×45 —
   the resolution the coordinates are given in);
3. join to the CAVE **`excitatory_neuron_label`** table to get the population tag.

The result ships as `../../data/psi_ihn_feather/psi_ihn_subtypes.csv`, so the figure
notebooks run offline. Only rerun this if the feather set changes.

The five tags map to the published populations as:

| tag | population |
| --- | --- |
| `IN for adhtmr` | PSI-IhN^Aδ-HTMR (n = 9) |
| `islet-chtmr` | Islet-PSI-IhN^C-HTMR/Heat (n = 6) |
| `islet-cltmr` | Islet-PSI-IhN^C-LTMR (n = 6) |
| `islet-adltmr` | Islet-PSI-IhN^Aδ-LTMR (n = 6) |
| `IN for abltmr` | PSI-IhN^Aβ-LTMR (n = 7) |

`excitatory_neuron_label` also carries an `IN for vertical cell` tag (n = 5); those neurons
have no feather files here and are not part of Figure 4.


In [ ]:
import glob
import os
import re
from pathlib import Path

import caveclient
import cloudvolume as cv
import numpy as np
import pandas as pd

MATERIALIZATION = 669
MIP = (32, 32, 45)
FEATHER_DIR = Path("../../data/psi_ihn_feather")

client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr")
if MATERIALIZATION is not None:
    client.version = MATERIALIZATION

seg = cv.CloudVolume(
    "graphene://https://cave.fanc-fly.com/segmentation/table/wclee_mouse_spinalcord_cltmr",
    mip=MIP, use_https=True, agglomerate=True, progress=False, fill_missing=True,
)

label = client.materialize.query_table("excitatory_neuron_label")
root_to_tag = dict(zip(label.pt_root_id.astype(int), label.tag))
print(label.tag.value_counts().to_string())


In [ ]:
rows = []
for f in sorted(glob.glob(str(FEATHER_DIR / "dendritic_inputs" / "*.feather"))):
    x, y, z = map(int, re.search(r"(\d+)_(\d+)_(\d+)\.feather", os.path.basename(f)).groups())
    root_id = int(np.asarray(seg[x, y, z]).squeeze())
    rows.append({"coord": f"{x}_{y}_{z}", "root_id": root_id,
                 "tag": root_to_tag.get(root_id)})

df = pd.DataFrame(rows)
missing = df.tag.isna().sum()
print(f"{len(df)} feather files, {missing} without a tag")
print(df.tag.value_counts().to_string())

df.to_csv(FEATHER_DIR / "psi_ihn_subtypes.csv", index=False)
print(f"\nwrote {FEATHER_DIR / 'psi_ihn_subtypes.csv'}")
